SIMPLE

CÁCH 1: Programmatic style
CÁCH 2: Column Expression style
CÁCH 3: SQL style

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession. \
builder. \
master("local[*]"). \
appName("Practice df 4"). \
getOrCreate()

In [2]:
df = spark.read \
.format("csv") \
.option ("inferSchema", "true") \
.option("header", "true") \
.load("D:\Learn-spark\learn-spark-maide\invoices.csv")

In [3]:
df.show()

+---------+---------+-----------+--------+-----------+---------+----------+---------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|  Country|
+---------+---------+-----------+--------+-----------+---------+----------+---------+
|  INV1000|     NULL|  Product 0|      20| 2024-01-04|    99.85|     66714|  Germany|
|  INV1000|     NULL|Product 0-1|      10| 2024-01-04|    99.85|     66714|  Germany|
|  INV1001|   STK802|  Product 1|      15| 2024-11-26|    81.42|     61694|   France|
|  INV1002|     NULL|  Product 2|      18| 2024-09-24|    89.16|     32208|   France|
|  INV1002|     NULL|Product 2-2|      15| 2024-09-24|    89.16|     32208|   France|
|  INV1003|     NULL|  Product 3|       5| 2024-09-14|    68.33|     72547|       UK|
|  INV1004|     NULL|  Product 4|       8| 2024-05-02|     56.4|     63055|      USA|
|  INV1005|   STK712|  Product 5|      19| 2024-12-09|    22.49|     95488|   France|
|  INV1006|   STK202|  Product 6|      19| 2024-01-18|

CÁCH 1

1. Simple: tổng số dòng

In [4]:
from pyspark.sql.functions import *

In [5]:
#CÁCH 1:
df.select(count("*").alias("total_rows")).show()

+----------+
|total_rows|
+----------+
|       102|
+----------+



SIMPLE: tổng số lượng hóa đơn

In [6]:

df.select(count("*").alias("total_rows"),
          count_distinct("InvoiceNo").alias("invoice_unique")) \
.show()

+----------+--------------+
|total_rows|invoice_unique|
+----------+--------------+
|       102|           100|
+----------+--------------+



SIMPLE: tổng số lượng hàng hóa

In [7]:
df.select(count("*").alias("total_rows"),
          count_distinct("InvoiceNo").alias("invoice_unique"),
          sum("Quantity").alias("total_quantity")) \
.show()

+----------+--------------+--------------+
|total_rows|invoice_unique|total_quantity|
+----------+--------------+--------------+
|       102|           100|          1088|
+----------+--------------+--------------+



SIMPLE-Tính giá trị trung bình của unit price

In [8]:
# df.select(count("*").alias("total_rows"),
#           count_distinct("InvoiceNo").alias("invoice_unique"),
#           sum("Quantity").alias("total_quantity"),
#           avg("UnitPrice").alias("avg_price")) \
# .show()

## Lay 1 so sau dau phay
df.select(count("*").alias("total_rows"),
          count_distinct("InvoiceNo").alias("invoice_unique"),
          sum("Quantity").alias("total_quantity"),
          round(avg("UnitPrice"), 1).alias("avg_price")) \
.show()

+----------+--------------+--------------+---------+
|total_rows|invoice_unique|total_quantity|avg_price|
+----------+--------------+--------------+---------+
|       102|           100|          1088|     53.2|
+----------+--------------+--------------+---------+



CÁCH 2

In [9]:
# df.selectExpr("count(*) as total_rows",
#                "count(distinct (InvoiceNo)) as invoice_unique",
#                 "sum(Quantity) as total_quantity",
#                 "avg(UnitPrice) as avg_price").show()

df.selectExpr("count(*) as total_rows",
               "count(distinct (InvoiceNo)) as invoice_unique",
                "sum(Quantity) as total_quantity",
                "round(avg(UnitPrice),2) as avg_price").show()


+----------+--------------+--------------+---------+
|total_rows|invoice_unique|total_quantity|avg_price|
+----------+--------------+--------------+---------+
|       102|           100|          1088|    53.18|
+----------+--------------+--------------+---------+



CÁCH 3- SQL

In [10]:
df.createOrReplaceTempView("orders")

In [11]:
spark.sql("select count(*) as total_rows, count(distinct(InvoiceNo)) as invoice_unique, sum(Quantity) as total_quantity, avg(UnitPrice) as avg_price from orders").show()

+----------+--------------+--------------+-----------------+
|total_rows|invoice_unique|total_quantity|        avg_price|
+----------+--------------+--------------+-----------------+
|       102|           100|          1088|53.18460784313723|
+----------+--------------+--------------+-----------------+



TYPE: GROUPING
3 CÁCH

CÁCH 1

In [12]:
df.show(5)

+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|  INV1000|     NULL|  Product 0|      20| 2024-01-04|    99.85|     66714|Germany|
|  INV1000|     NULL|Product 0-1|      10| 2024-01-04|    99.85|     66714|Germany|
|  INV1001|   STK802|  Product 1|      15| 2024-11-26|    81.42|     61694| France|
|  INV1002|     NULL|  Product 2|      18| 2024-09-24|    89.16|     32208| France|
|  INV1002|     NULL|Product 2-2|      15| 2024-09-24|    89.16|     32208| France|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
only showing top 5 rows



Tổng số lượng hàng hóa của mỗi quốc gia

CÁCH 1

In [13]:
df.groupBy("Country") \
    .agg(sum("Quantity").alias("total_quantity")).show()

+---------+--------------+
|  Country|total_quantity|
+---------+--------------+
|  Germany|           162|
|   France|           181|
|      USA|           129|
|       UK|           217|
|   Canada|           211|
|Australia|           188|
+---------+--------------+



Tổng doanh số bán hàng của mỗi quốc gia

In [16]:
df.groupBy("Country") \
    .agg(sum("Quantity").alias("total_quantity"),
    round(sum(df.Quantity  * df.UnitPrice), 2).alias("total_amount")) \
.show()

+---------+--------------+------------+
|  Country|total_quantity|total_amount|
+---------+--------------+------------+
|  Germany|           162|     8494.55|
|   France|           181|    10970.44|
|      USA|           129|     5688.58|
|       UK|           217|    11967.68|
|   Canada|           211|     8797.13|
|Australia|           188|    10530.25|
+---------+--------------+------------+



CÁCH 2

In [18]:
df.groupBy("Country") \
    .agg(expr("sum (Quantity) as total_quantity")).show()

+---------+--------------+
|  Country|total_quantity|
+---------+--------------+
|  Germany|           162|
|   France|           181|
|      USA|           129|
|       UK|           217|
|   Canada|           211|
|Australia|           188|
+---------+--------------+



In [20]:
# df.groupBy("Country") \
#     .agg(expr("sum (Quantity) as total_quantity"),
#          expr("sum(Quantity * UnitPrice) as total_amount")) \
# .show()

df.groupBy("Country") \
    .agg(expr("sum (Quantity) as total_quantity"),
         expr("round(sum(Quantity * UnitPrice),2) as total_amount")) \
.show()

+---------+--------------+------------+
|  Country|total_quantity|total_amount|
+---------+--------------+------------+
|  Germany|           162|     8494.55|
|   France|           181|    10970.44|
|      USA|           129|     5688.58|
|       UK|           217|    11967.68|
|   Canada|           211|     8797.13|
|Australia|           188|    10530.25|
+---------+--------------+------------+



CÁCH 3

In [21]:
df.createOrReplaceTempView("orders")

In [23]:
spark.sql("select Country, sum(Quantity) as total_quantity, round(sum(Quantity * UnitPrice), 2) as total_amount from orders group by country").show()                                                        

+---------+--------------+------------+
|  Country|total_quantity|total_amount|
+---------+--------------+------------+
|  Germany|           162|     8494.55|
|   France|           181|    10970.44|
|      USA|           129|     5688.58|
|       UK|           217|    11967.68|
|   Canada|           211|     8797.13|
|Australia|           188|    10530.25|
+---------+--------------+------------+

